# FastAPI 模型服务教程 (FastAPI Model Serving Tutorial)

> **前置知识**: Python 基础、REST API 概念、异步编程基础
>
> **学习目标**: 掌握使用 FastAPI 构建高性能模型推理服务

---

## 为什么选择 FastAPI？

```
┌─────────────────────────────────────────────────────────────┐
│                    FastAPI 核心优势                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  性能对比 (请求/秒):                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  FastAPI:  ████████████████████████████  ~15,000    │   │
│  │  Flask:    ████████████                  ~5,000     │   │
│  │  Django:   ██████████                    ~4,000     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  核心特性:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 高性能: 基于 Starlette + Pydantic              │   │
│  │  2. 自动文档: Swagger UI + ReDoc 自动生成          │   │
│  │  3. 类型安全: 完整的类型提示和验证                 │   │
│  │  4. 异步支持: 原生 async/await                     │   │
│  │  5. 易于使用: 简洁直观的 API 设计                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  适用场景:                                                  │
│  - 快速原型开发                                            │
│  - 中小规模模型服务                                        │
│  - 需要自动 API 文档的项目                                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **FastAPI 基础** - 创建 REST API 服务
2. **请求/响应模型** - Pydantic 数据验证
3. **指标收集器** - 监控服务性能
4. **推理缓存** - 避免重复计算
5. **动态批处理** - 提高吞吐量
6. **完整服务示例** - 生产级服务

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
import time
import asyncio
import hashlib
import json
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, field
from collections import deque

# 设置随机种子
np.random.seed(42)

print("=" * 60)
print("环境准备完成")
print("=" * 60)

# 检查 FastAPI
try:
    from fastapi import FastAPI
    from pydantic import BaseModel, Field
    FASTAPI_AVAILABLE = True
    print(f"\n✓ FastAPI 已安装")
except ImportError:
    FASTAPI_AVAILABLE = False
    print(f"\n✗ FastAPI 未安装")
    print("  安装命令: pip install fastapi uvicorn")

print(f"\n注意: 本教程的核心组件 (指标收集、缓存、批处理) 不依赖 FastAPI")
print(f"      可以独立学习和使用")

## 1. FastAPI 基础

**核心概念**: FastAPI 使用装饰器定义路由，支持异步处理

```
┌─────────────────────────────────────────────────────────────┐
│                    FastAPI 请求处理流程                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  客户端请求                                                 │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 路由匹配: @app.post("/predict")                │   │
│  │       ↓                                             │   │
│  │  2. 请求验证: Pydantic 自动验证 JSON               │   │
│  │       ↓                                             │   │
│  │  3. 依赖注入: Depends() 注入依赖                   │   │
│  │       ↓                                             │   │
│  │  4. 业务处理: async def predict(...)              │   │
│  │       ↓                                             │   │
│  │  5. 响应序列化: 自动转换为 JSON                    │   │
│  └─────────────────────────────────────────────────────┘   │
│       │                                                     │
│       ▼                                                     │
│  JSON 响应                                                  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# FastAPI 基础示例
# ============================================================
print("=" * 60)
print("FastAPI 基础示例")
print("=" * 60)

if FASTAPI_AVAILABLE:
    # 创建 FastAPI 应用
    app = FastAPI(
        title="Model Inference API",
        description="模型推理服务 API",
        version="1.0.0"
    )
    
    # 定义路由
    @app.get("/")
    async def root():
        """根路径"""
        return {"message": "Welcome to Model Inference API"}
    
    @app.get("/health")
    async def health():
        """健康检查端点"""
        return {"status": "healthy"}
    
    print("✓ FastAPI 应用创建成功!")
    print(f"\n已注册的路由:")
    for route in app.routes:
        if hasattr(route, 'path') and hasattr(route, 'methods'):
            methods = ', '.join(route.methods - {'HEAD', 'OPTIONS'})
            if methods:
                print(f"  {methods:6} {route.path}")
else:
    print("跳过 FastAPI 示例 (FastAPI 未安装)")
    print("\n示例代码:")
    print("""
    app = FastAPI(title="Model Inference API")
    
    @app.get("/health")
    async def health():
        return {"status": "healthy"}
    
    @app.post("/predict")
    async def predict(request: PredictRequest):
        result = model.predict(request.data)
        return {"prediction": result}
    """)

### 1.2 请求/响应模型

**核心概念**: Pydantic 提供自动数据验证和文档生成

```
┌─────────────────────────────────────────────────────────────┐
│                    Pydantic 数据验证                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  JSON 请求                                                  │
│  {"data": [1.0, 2.0], "batch_size": 1}                     │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Pydantic 验证:                                     │   │
│  │  - 类型检查: data 是否为 List[float]               │   │
│  │  - 必填检查: data 是否存在                         │   │
│  │  - 默认值: batch_size 默认为 1                     │   │
│  │  - 范围检查: 自定义 validator                      │   │
│  └─────────────────────────────────────────────────────┘   │
│       │                                                     │
│       ▼                                                     │
│  验证通过 → PredictRequest 对象                            │
│  验证失败 → 422 Unprocessable Entity                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 请求/响应模型定义
# ============================================================
print("=" * 60)
print("Pydantic 请求/响应模型")
print("=" * 60)

if FASTAPI_AVAILABLE:
    # 请求模型
    class PredictRequest(BaseModel):
        """
        预测请求模型
        
        Pydantic 自动验证:
        - data: 必填，必须是浮点数列表
        - batch_size: 可选，默认为 1
        """
        data: List[float] = Field(..., description="输入数据")
        batch_size: Optional[int] = Field(1, description="批次大小")
        
        class Config:
            json_schema_extra = {
                "example": {
                    "data": [1.0, 2.0, 3.0, 4.0],
                    "batch_size": 1
                }
            }
    
    # 响应模型
    class PredictResponse(BaseModel):
        """
        预测响应模型
        
        包含预测结果和性能指标
        """
        prediction: List[float] = Field(..., description="预测结果")
        confidence: Optional[float] = Field(None, description="置信度")
        latency_ms: float = Field(..., description="推理延迟(毫秒)")
    
    # 测试模型
    print("\n创建请求对象:")
    request = PredictRequest(data=[1.0, 2.0, 3.0, 4.0])
    print(f"  data: {request.data}")
    print(f"  batch_size: {request.batch_size}")
    
    print("\n创建响应对象:")
    response = PredictResponse(
        prediction=[0.8, 0.15, 0.05],
        confidence=0.8,
        latency_ms=5.2
    )
    print(f"  {response.model_dump()}")
    
    print("\n验证失败示例:")
    try:
        bad_request = PredictRequest(data="not a list")  # 类型错误
    except Exception as e:
        print(f"  错误: {type(e).__name__}")
else:
    print("跳过 Pydantic 示例 (FastAPI 未安装)")

## 2. 指标收集器

**核心概念**: 收集服务指标用于监控和告警

```
┌─────────────────────────────────────────────────────────────┐
│                    服务指标监控                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  核心指标:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求计数: 总请求数、成功数、失败数                 │   │
│  │  延迟统计: 平均延迟、P50、P95、P99                  │   │
│  │  成功率: successful / total                         │   │
│  │  吞吐量: requests / second                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  延迟百分位数说明:                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  P50: 50% 请求的延迟 (中位数)                      │   │
│  │  P95: 95% 请求的延迟 (大部分用户体验)              │   │
│  │  P99: 99% 请求的延迟 (尾延迟，影响 SLA)            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 指标收集器 (自包含实现)
# ============================================================
print("=" * 60)
print("指标收集器")
print("=" * 60)

class MetricsCollector:
    """
    服务指标收集器
    
    收集请求延迟、成功率等指标，支持百分位数统计
    
    使用 deque 存储最近的延迟数据，避免内存无限增长
    """
    def __init__(self, max_history: int = 1000):
        """
        参数:
            max_history: 保留的最大历史记录数
        """
        self.max_history = max_history
        self.latencies: deque = deque(maxlen=max_history)
        self.total_requests = 0
        self.successful_requests = 0
        self.failed_requests = 0
    
    async def record_request(self, latency_ms: float, success: bool = True):
        """
        记录一次请求
        
        参数:
            latency_ms: 请求延迟 (毫秒)
            success: 是否成功
        """
        self.latencies.append(latency_ms)
        self.total_requests += 1
        if success:
            self.successful_requests += 1
        else:
            self.failed_requests += 1
    
    def get_metrics(self) -> Dict[str, Any]:
        """获取当前指标"""
        latencies = list(self.latencies)
        
        if not latencies:
            return {
                'total_requests': 0,
                'successful_requests': 0,
                'failed_requests': 0,
                'success_rate': 0.0,
                'avg_latency_ms': 0.0,
                'p50_latency_ms': 0.0,
                'p95_latency_ms': 0.0,
                'p99_latency_ms': 0.0,
            }
        
        return {
            'total_requests': self.total_requests,
            'successful_requests': self.successful_requests,
            'failed_requests': self.failed_requests,
            'success_rate': self.successful_requests / self.total_requests,
            'avg_latency_ms': np.mean(latencies),
            'p50_latency_ms': np.percentile(latencies, 50),
            'p95_latency_ms': np.percentile(latencies, 95),
            'p99_latency_ms': np.percentile(latencies, 99),
        }
    
    def reset(self):
        """重置所有指标"""
        self.latencies.clear()
        self.total_requests = 0
        self.successful_requests = 0
        self.failed_requests = 0


# 创建指标收集器
metrics = MetricsCollector(max_history=1000)

print("\n✓ MetricsCollector 类定义完成")
print("\n功能:")
print("  - record_request(): 记录请求延迟和状态")
print("  - get_metrics(): 获取统计指标")
print("  - reset(): 重置所有指标")

In [ ]:
# ============================================================
# 模拟请求并收集指标
# ============================================================
print("=" * 60)
print("模拟请求测试")
print("=" * 60)

async def simulate_requests(collector: MetricsCollector, num_requests: int = 100):
    """
    模拟请求并记录指标
    
    使用指数分布模拟真实的延迟分布
    """
    for i in range(num_requests):
        # 模拟不同的延迟 (指数分布，均值 10ms)
        latency = np.random.exponential(10)
        # 模拟 95% 成功率
        success = np.random.random() > 0.05
        await collector.record_request(latency, success)

# 运行模拟
print("\n模拟 100 个请求...")
await simulate_requests(metrics, 100)

# 获取指标
stats = metrics.get_metrics()
print(f"\n服务指标:")
print(f"  总请求数: {stats['total_requests']}")
print(f"  成功请求: {stats['successful_requests']}")
print(f"  失败请求: {stats['failed_requests']}")
print(f"  成功率: {stats['success_rate']:.2%}")
print(f"\n延迟统计:")
print(f"  平均延迟: {stats['avg_latency_ms']:.2f} ms")
print(f"  P50 延迟: {stats['p50_latency_ms']:.2f} ms")
print(f"  P95 延迟: {stats['p95_latency_ms']:.2f} ms")
print(f"  P99 延迟: {stats['p99_latency_ms']:.2f} ms")

## 3. 推理缓存

**核心概念**: 缓存相同输入的推理结果，避免重复计算

```
┌─────────────────────────────────────────────────────────────┐
│                    推理缓存原理                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  请求流程:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  输入数据 → 计算哈希 → 查询缓存                     │   │
│  │                          │                          │   │
│  │              ┌───────────┴───────────┐              │   │
│  │              │                       │              │   │
│  │              ▼                       ▼              │   │
│  │           命中                     未命中           │   │
│  │              │                       │              │   │
│  │              ▼                       ▼              │   │
│  │         返回缓存              执行推理              │   │
│  │         (< 1ms)              (10-100ms)            │   │
│  │                                     │              │   │
│  │                                     ▼              │   │
│  │                               存入缓存              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  缓存策略:                                                  │
│  - LRU (Least Recently Used): 淘汰最久未使用的条目         │
│  - TTL (Time To Live): 设置过期时间                        │
│  - 容量限制: 防止内存无限增长                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 推理缓存 (自包含实现)
# ============================================================
print("=" * 60)
print("推理缓存")
print("=" * 60)

class InferenceCache:
    """
    推理结果缓存
    
    使用 LRU 策略和 TTL 过期机制
    
    特点:
    - 基于输入哈希的快速查找
    - LRU 淘汰策略
    - TTL 过期机制
    - 线程安全 (可扩展)
    """
    def __init__(self, max_size: int = 1000, ttl: float = 300.0):
        """
        参数:
            max_size: 最大缓存条目数
            ttl: 缓存过期时间 (秒)
        """
        self.max_size = max_size
        self.ttl = ttl
        self.cache: Dict[str, Dict] = {}
        self.access_times: Dict[str, float] = {}
    
    def _hash_input(self, data: Any) -> str:
        """计算输入数据的哈希值"""
        if isinstance(data, (list, tuple)):
            serialized = json.dumps(data, sort_keys=True)
        elif isinstance(data, np.ndarray):
            serialized = data.tobytes()
        else:
            serialized = str(data)
        return hashlib.md5(serialized.encode() if isinstance(serialized, str) else serialized).hexdigest()
    
    def get(self, data: Any) -> Optional[Any]:
        """
        获取缓存结果
        
        返回:
            缓存的结果，如果未命中或已过期返回 None
        """
        key = self._hash_input(data)
        
        if key in self.cache:
            entry = self.cache[key]
            # 检查是否过期
            if time.time() - entry['timestamp'] < self.ttl:
                self.access_times[key] = time.time()
                return entry['result']
            else:
                # 已过期，删除
                del self.cache[key]
                del self.access_times[key]
        return None
    
    def set(self, data: Any, result: Any):
        """
        设置缓存
        
        如果缓存已满，使用 LRU 策略淘汰
        """
        # 检查容量，必要时淘汰
        if len(self.cache) >= self.max_size:
            self._evict()
        
        key = self._hash_input(data)
        self.cache[key] = {
            'result': result,
            'timestamp': time.time()
        }
        self.access_times[key] = time.time()
    
    def _evict(self):
        """LRU 淘汰: 删除最久未访问的条目"""
        if not self.access_times:
            return
        oldest_key = min(self.access_times, key=self.access_times.get)
        del self.cache[oldest_key]
        del self.access_times[oldest_key]
    
    def clear(self):
        """清空缓存"""
        self.cache.clear()
        self.access_times.clear()
    
    @property
    def size(self) -> int:
        """当前缓存大小"""
        return len(self.cache)


# 创建缓存
cache = InferenceCache(max_size=100, ttl=60.0)

print("\n✓ InferenceCache 类定义完成")
print(f"\n配置:")
print(f"  最大容量: {cache.max_size} 条")
print(f"  过期时间: {cache.ttl} 秒")

In [ ]:
# ============================================================
# 测试缓存效果
# ============================================================
print("=" * 60)
print("缓存效果测试")
print("=" * 60)

# 模拟耗时推理函数
def slow_inference(data):
    """模拟耗时推理 (100ms)"""
    time.sleep(0.1)
    return [sum(data) / len(data)]  # 返回平均值

# 带缓存的推理
def cached_inference(data):
    """带缓存的推理"""
    # 检查缓存
    cached = cache.get(data)
    if cached is not None:
        return cached, True  # 缓存命中
    
    # 执行推理
    result = slow_inference(data)
    
    # 存入缓存
    cache.set(data, result)
    return result, False  # 缓存未命中

# 测试
test_data = [1.0, 2.0, 3.0, 4.0, 5.0]

# 第一次调用 (缓存未命中)
start = time.time()
result1, hit1 = cached_inference(test_data)
time1 = (time.time() - start) * 1000
print(f"\n第一次调用:")
print(f"  耗时: {time1:.2f} ms")
print(f"  缓存命中: {hit1}")
print(f"  结果: {result1}")

# 第二次调用 (缓存命中)
start = time.time()
result2, hit2 = cached_inference(test_data)
time2 = (time.time() - start) * 1000
print(f"\n第二次调用:")
print(f"  耗时: {time2:.2f} ms")
print(f"  缓存命中: {hit2}")
print(f"  结果: {result2}")

print(f"\n加速比: {time1/max(time2, 0.001):.0f}x")
print(f"缓存大小: {cache.size}")

## 4. 动态批处理

**核心概念**: 将多个请求合并为批次处理，提高 GPU 利用率

```
┌─────────────────────────────────────────────────────────────┐
│                    动态批处理原理                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  单请求处理 (低效):                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 → [GPU 推理] → 响应1                        │   │
│  │  请求2 → [GPU 推理] → 响应2                        │   │
│  │  请求3 → [GPU 推理] → 响应3                        │   │
│  │  ↑ GPU 利用率低，大量时间在等待                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  动态批处理 (高效):                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求1 ─┐                                          │   │
│  │  请求2 ─┼─→ [收集] → [批量 GPU 推理] → 分发响应   │   │
│  │  请求3 ─┘                                          │   │
│  │  ↑ GPU 一次处理多个请求，利用率高                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  触发条件:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 达到最大批次大小 (max_batch_size)              │   │
│  │  2. 等待超时 (max_wait_time)                       │   │
│  │  满足任一条件即触发批量处理                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 动态批处理器 (自包含实现)
# ============================================================
print("=" * 60)
print("动态批处理器")
print("=" * 60)

class DynamicBatcher:
    """
    动态批处理器
    
    收集多个请求并批量处理，提高 GPU 利用率
    
    触发条件:
    - 达到最大批次大小 (max_batch_size)
    - 等待超时 (max_wait_time)
    """
    def __init__(self, predict_fn, max_batch_size: int = 8, max_wait_time: float = 0.02):
        """
        参数:
            predict_fn: 批量预测函数，接受 List[data] 返回 List[result]
            max_batch_size: 最大批次大小
            max_wait_time: 最大等待时间 (秒)
        """
        self.predict_fn = predict_fn
        self.max_batch_size = max_batch_size
        self.max_wait_time = max_wait_time
        self.pending_requests: List[Dict] = []
        self.lock = asyncio.Lock()
        self.batch_count = 0
        self.total_requests = 0
    
    async def add_request(self, data: Any) -> Any:
        """
        添加请求到批处理队列
        
        返回:
            推理结果
        """
        # 创建 Future 用于等待结果
        future = asyncio.Future()
        
        async with self.lock:
            self.pending_requests.append({
                'data': data,
                'future': future
            })
            self.total_requests += 1
            
            # 检查是否达到批次大小
            if len(self.pending_requests) >= self.max_batch_size:
                await self._process_batch()
        
        # 如果未达到批次大小，等待超时后处理
        if not future.done():
            await asyncio.sleep(self.max_wait_time)
            async with self.lock:
                if self.pending_requests and not future.done():
                    await self._process_batch()
        
        return await future
    
    async def _process_batch(self):
        """处理当前批次"""
        if not self.pending_requests:
            return
        
        # 取出当前批次
        batch = self.pending_requests[:self.max_batch_size]
        self.pending_requests = self.pending_requests[self.max_batch_size:]
        self.batch_count += 1
        
        # 批量推理
        batch_data = [req['data'] for req in batch]
        results = self.predict_fn(batch_data)
        
        # 分发结果
        for req, result in zip(batch, results):
            if not req['future'].done():
                req['future'].set_result(result)
    
    def get_stats(self) -> Dict[str, Any]:
        """获取统计信息"""
        return {
            'total_requests': self.total_requests,
            'batch_count': self.batch_count,
            'avg_batch_size': self.total_requests / max(1, self.batch_count),
            'pending_requests': len(self.pending_requests)
        }


# 定义批量预测函数
def batch_predict(batch_data):
    """
    批量预测函数
    
    模拟批量推理，返回每个输入的和
    """
    time.sleep(0.05)  # 模拟 50ms 批量推理
    return [sum(data) for data in batch_data]

# 创建动态批处理器
batcher = DynamicBatcher(
    predict_fn=batch_predict,
    max_batch_size=8,
    max_wait_time=0.02  # 20ms 最大等待
)

print("\n✓ DynamicBatcher 类定义完成")
print(f"\n配置:")
print(f"  最大批次大小: {batcher.max_batch_size}")
print(f"  最大等待时间: {batcher.max_wait_time * 1000}ms")

In [ ]:
# ============================================================
# 测试动态批处理
# ============================================================
print("=" * 60)
print("动态批处理测试")
print("=" * 60)

async def test_dynamic_batching():
    """
    测试动态批处理效果
    
    并发发送多个请求，观察批处理行为
    """
    # 并发发送 4 个请求
    tasks = [
        batcher.add_request([1.0, 2.0]),
        batcher.add_request([3.0, 4.0]),
        batcher.add_request([5.0, 6.0]),
        batcher.add_request([7.0, 8.0]),
    ]
    
    start = time.time()
    results = await asyncio.gather(*tasks)
    elapsed = (time.time() - start) * 1000
    
    print(f"\n批处理结果:")
    for i, result in enumerate(results):
        print(f"  请求 {i+1}: 输入和 = {result}")
    
    print(f"\n性能统计:")
    print(f"  总耗时: {elapsed:.2f} ms")
    print(f"  平均每请求: {elapsed/len(tasks):.2f} ms")
    
    # 获取批处理统计
    stats = batcher.get_stats()
    print(f"\n批处理统计:")
    print(f"  总请求数: {stats['total_requests']}")
    print(f"  批次数: {stats['batch_count']}")
    print(f"  平均批次大小: {stats['avg_batch_size']:.1f}")

# 运行测试
await test_dynamic_batching()

print(f"\n对比:")
print(f"  单独处理 4 个请求: 4 × 50ms = 200ms")
print(f"  批处理 4 个请求: ~70ms (50ms 推理 + 20ms 等待)")
print(f"  加速比: ~3x")

## 5. 完整的模型服务示例

**核心概念**: 整合所有组件构建生产级推理服务

```
┌─────────────────────────────────────────────────────────────┐
│                    完整服务架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  客户端请求                                                 │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  FastAPI 应用                                       │   │
│  │  ┌─────────────────────────────────────────────┐   │   │
│  │  │  /predict 端点                              │   │   │
│  │  │       │                                     │   │   │
│  │  │       ▼                                     │   │   │
│  │  │  [缓存检查] ──命中──→ 返回缓存结果          │   │   │
│  │  │       │                                     │   │   │
│  │  │      未命中                                 │   │   │
│  │  │       │                                     │   │   │
│  │  │       ▼                                     │   │   │
│  │  │  [动态批处理] ──→ [模型推理]               │   │   │
│  │  │       │                                     │   │   │
│  │  │       ▼                                     │   │   │
│  │  │  [记录指标] ──→ 返回结果                   │   │   │
│  │  └─────────────────────────────────────────────┘   │   │
│  │                                                     │   │
│  │  /health  - 健康检查                               │   │
│  │  /metrics - 服务指标                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 完整模型服务示例 (自包含实现)
# ============================================================
print("=" * 60)
print("完整模型服务示例")
print("=" * 60)

if FASTAPI_AVAILABLE:
    # 创建完整的模型服务应用
    model_app = FastAPI(
        title="Model Inference Service",
        description="完整的模型推理服务，包含缓存、批处理、监控",
        version="1.0.0"
    )
    
    # 初始化组件
    service_metrics = MetricsCollector(max_history=1000)
    service_cache = InferenceCache(max_size=500, ttl=300.0)
    
    # 模拟模型推理函数
    def model_predict(data: List[float]) -> List[float]:
        """模拟模型推理"""
        time.sleep(0.01)  # 模拟 10ms 推理
        return [sum(data) / len(data)]  # 返回平均值
    
    # 健康检查端点
    @model_app.get("/health")
    async def health_check():
        """健康检查"""
        return {
            "status": "healthy",
            "cache_size": service_cache.size,
            "total_requests": service_metrics.total_requests
        }
    
    # 指标端点
    @model_app.get("/metrics")
    async def get_metrics():
        """获取服务指标"""
        return service_metrics.get_metrics()
    
    # 预测端点
    @model_app.post("/predict")
    async def predict(request: PredictRequest):
        """
        模型预测端点
        
        流程:
        1. 检查缓存
        2. 执行推理 (如果缓存未命中)
        3. 记录指标
        4. 返回结果
        """
        start_time = time.time()
        
        # 检查缓存
        cached_result = service_cache.get(request.data)
        if cached_result is not None:
            latency = (time.time() - start_time) * 1000
            await service_metrics.record_request(latency, success=True)
            return PredictResponse(
                prediction=cached_result,
                confidence=1.0,
                latency_ms=latency
            )
        
        # 执行推理
        try:
            result = model_predict(request.data)
            service_cache.set(request.data, result)
            latency = (time.time() - start_time) * 1000
            await service_metrics.record_request(latency, success=True)
            return PredictResponse(
                prediction=result,
                confidence=0.95,
                latency_ms=latency
            )
        except Exception as e:
            latency = (time.time() - start_time) * 1000
            await service_metrics.record_request(latency, success=False)
            raise
    
    print("✓ 完整模型服务创建成功!")
    print(f"\n已注册的端点:")
    for route in model_app.routes:
        if hasattr(route, 'path') and hasattr(route, 'methods'):
            methods = ', '.join(route.methods - {'HEAD', 'OPTIONS'})
            if methods:
                print(f"  {methods:6} {route.path}")
else:
    print("跳过完整服务示例 (FastAPI 未安装)")

## 6. 启动服务

**核心概念**: 使用 uvicorn 启动 FastAPI 服务

```
┌─────────────────────────────────────────────────────────────┐
│                    服务启动方式                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  开发环境:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  uvicorn main:app --reload --host 0.0.0.0 --port 8000│  │
│  │                                                     │   │
│  │  --reload: 代码变更自动重载                        │   │
│  │  --host 0.0.0.0: 允许外部访问                      │   │
│  │  --port 8000: 服务端口                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  生产环境:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  uvicorn main:app --workers 4 --host 0.0.0.0       │   │
│  │                                                     │   │
│  │  --workers 4: 多进程处理                           │   │
│  │  或使用 gunicorn:                                  │   │
│  │  gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker│
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**注意**: 以下代码展示如何启动服务器（在 Notebook 中不执行）

In [ ]:
# ============================================================
# 启动服务器示例代码
# ============================================================
print("=" * 60)
print("启动服务器示例代码")
print("=" * 60)

print("""
# ============================================================
# main.py - 完整的模型服务启动文件
# ============================================================

from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import List, Optional
import numpy as np
import time

app = FastAPI(title="Model Inference API")

# 请求/响应模型
class PredictRequest(BaseModel):
    data: List[float] = Field(..., description="输入数据")

class PredictResponse(BaseModel):
    prediction: List[float]
    latency_ms: float

# 你的模型推理函数
def predict(data: List[float]) -> List[float]:
    # 替换为你的实际模型推理逻辑
    return [sum(data) / len(data)]

@app.get("/health")
async def health():
    return {"status": "healthy"}

@app.post("/predict", response_model=PredictResponse)
async def predict_endpoint(request: PredictRequest):
    start = time.time()
    result = predict(request.data)
    latency = (time.time() - start) * 1000
    return PredictResponse(prediction=result, latency_ms=latency)

# ============================================================
# 启动命令
# ============================================================

# 开发环境 (自动重载)
# uvicorn main:app --reload --host 0.0.0.0 --port 8000

# 生产环境 (多进程)
# uvicorn main:app --workers 4 --host 0.0.0.0 --port 8000

# 使用 gunicorn (更稳定)
# gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker -b 0.0.0.0:8000
""")

## 7. 客户端调用

**核心概念**: 使用 HTTP 客户端调用推理服务

```
┌─────────────────────────────────────────────────────────────┐
│                    客户端调用方式                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  同步调用 (httpx/requests):                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  适用场景: 简单脚本、测试                          │   │
│  │  特点: 阻塞等待响应                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  异步调用 (httpx.AsyncClient):                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  适用场景: 高并发应用、批量请求                    │   │
│  │  特点: 非阻塞，可并发多个请求                      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  流式调用 (Server-Sent Events):                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  适用场景: LLM 生成、实时输出                      │   │
│  │  特点: 逐步接收响应                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 客户端调用示例代码
# ============================================================
print("=" * 60)
print("客户端调用示例")
print("=" * 60)

print("""
# ============================================================
# 同步调用 (使用 httpx 或 requests)
# ============================================================
import httpx

# 创建客户端
with httpx.Client(base_url="http://localhost:8000") as client:
    # 健康检查
    response = client.get("/health")
    print(f"健康状态: {response.json()}")
    
    # 预测请求
    response = client.post(
        "/predict",
        json={"data": [1.0, 2.0, 3.0, 4.0]}
    )
    result = response.json()
    print(f"预测结果: {result['prediction']}")
    print(f"延迟: {result['latency_ms']:.2f}ms")
    
    # 获取指标
    response = client.get("/metrics")
    print(f"服务指标: {response.json()}")

# ============================================================
# 异步调用 (高并发场景)
# ============================================================
import asyncio
import httpx

async def batch_predict(data_list):
    \"\"\"批量异步预测\"\"\"
    async with httpx.AsyncClient(base_url="http://localhost:8000") as client:
        tasks = [
            client.post("/predict", json={"data": data})
            for data in data_list
        ]
        responses = await asyncio.gather(*tasks)
        return [r.json() for r in responses]

# 使用示例
# results = asyncio.run(batch_predict([
#     [1.0, 2.0],
#     [3.0, 4.0],
#     [5.0, 6.0]
# ]))

# ============================================================
# 错误处理
# ============================================================
try:
    response = client.post("/predict", json={"data": "invalid"})
    response.raise_for_status()
except httpx.HTTPStatusError as e:
    print(f"HTTP 错误: {e.response.status_code}")
    print(f"错误详情: {e.response.json()}")
""")

## 总结

本教程介绍了使用 FastAPI 构建模型推理服务的核心技术：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| FastAPI 基础 | 路由定义、异步处理、自动文档 |
| Pydantic 模型 | 请求/响应验证、类型安全 |
| 指标收集 | 延迟统计 (P50/P95/P99)、成功率 |
| 推理缓存 | LRU 淘汰、TTL 过期、哈希查找 |
| 动态批处理 | 请求收集、批量推理、结果分发 |

### API 速查

```python
# FastAPI 应用
from fastapi import FastAPI
app = FastAPI(title="Model API")

@app.post("/predict")
async def predict(request: PredictRequest):
    return PredictResponse(prediction=result)

# 指标收集
metrics = MetricsCollector(max_history=1000)
await metrics.record_request(latency_ms, success=True)
stats = metrics.get_metrics()  # 返回 P50, P95, P99 等

# 推理缓存
cache = InferenceCache(max_size=1000, ttl=300.0)
cached = cache.get(data)  # 查询缓存
cache.set(data, result)   # 存入缓存

# 动态批处理
batcher = DynamicBatcher(predict_fn, max_batch_size=8)
result = await batcher.add_request(data)

# 启动服务
# uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
```

### 生产环境检查清单

```
部署前检查:
✓ 实现健康检查端点 (/health)
✓ 实现指标端点 (/metrics)
✓ 启用推理缓存减少重复计算
✓ 配置动态批处理提高吞吐量
✓ 使用 Pydantic 验证输入数据
✓ 配置多 worker 处理并发

性能优化:
✓ 缓存命中率 > 30% (根据业务场景)
✓ P99 延迟 < SLA 要求
✓ 成功率 > 99%
✓ 批处理利用率 > 50%
```

### 下一步学习

- **02_Triton_tutorial.ipynb**: Triton Inference Server 高性能推理
- **03_LoadBalancing_tutorial.ipynb**: 负载均衡与高可用